In [0]:

# 1. IMPORTS AND SETTINGS


from pyspark.sql import functions as F

# GitHub source folder
github_base_path = ("/Workspace/Users/ankurkamboj.ynr@gmail.com/""time_n_work_utilization_at_7trents/"
    "data_ingest_github/")

# Dimensions are available in Day 1 folder
source_path = github_base_path + "01_04_2026/"

# Target Bronze schema
target_schema = "severntrent.bronze_dev"

print("Dimension source:", source_path)
print("Target schema:", target_schema)


In [0]:

# 2. CREATE DIMENSION INGESTION FUNCTION


def dimension_ingestor(source_name):

    # 1. Define CSV file path
    file_path = source_path + source_name + ".csv"

    print("Reading:", file_path)

    # 2. Read CSV file
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    # 3. Add audit columns
    df = df.withColumn("source_file",F.lit(source_name + ".csv"))

    df = df.withColumn("load_date",F.to_date(F.lit("01_04_2026"), "dd_MM_yyyy"))

    df = df.withColumn("ingestion_ts",F.current_timestamp())

    df = df.withColumn("load_type",F.lit("FULL"))

    print("Total records:", df.count())

    return df

In [0]:

# 3. TEST DIMENSION INGESTION


df_people = dimension_ingestor("DimPeople")

df_people.show(5, truncate=False)

In [0]:

# 4. LOAD ALL DIMENSION TABLES TO BRONZE


# Source filenames and target table names
dimension_tables = {
    "DimPeople": "dim_people",
    "DimCounty": "dim_county",
    "DimCancellationReason": "dim_cancellation_reason",
    "DimShrinkageType": "dim_shrinkage_type"
}

# Process each dimension table
for source_name, table_name in dimension_tables.items():

    target_table = target_schema + "." + table_name

    # Check if table already exists
    if spark.catalog.tableExists(target_table):
        print(table_name, "already exists; skipping")

    else:
        # Read data using our reusable function
        df = dimension_ingestor(source_name)

        # Save as Delta table
        df.write \
            .format("delta") \
            .saveAsTable(target_table)

        print(table_name, "saved successfully")

In [0]:

# 5. VALIDATE ALL DIMENSION TABLES

for table_name in dimension_tables.values():

    target_table = target_schema + "." + table_name

    df = spark.table(target_table)

    print("\nTable:", target_table)
    print("Total records:", df.count())

    df.groupBy("load_date", "load_type").count().show()